sankey chart / Alluvial Diagram 

In [1]:
# imports
import pandas as pd
import matplotlib as plt
import math

In [2]:
# set display options for pandas
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [3]:
# constants
years = [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
normalizedColumns = ["fund_type", "fund_description", "department_description", "appropriation_authority_description",  "appropriation_account_description"]

In [4]:
# helper functions
def getRowAmount(row):
    if not pd.isna(row["_ordinance_amount_"]):
        return row["_ordinance_amount_"]
    elif not pd.isna(row["appropriation_ordinance"]):
        return row["appropriation_ordinance"]
    else:
        return row["amount"]

def getValueCounts(cols):
    return df[cols].value_counts().to_frame().reset_index()

def getAltNames(df, indexCol, nameCol):
    count = df[indexCol].value_counts().to_frame().reset_index()
    mult = count[count["count"] > 1]
    mult["names"] = mult.apply(lambda row: df[df[indexCol] == row[indexCol]][nameCol].to_list(), axis=1)
    return mult

def getAnnualBudget(df, year):
    return df[df["year"] == year]["amount"].sum()

In [21]:
# import all ordinance files and combine into table
df = pd.DataFrame()
for year in years:
    dfYear = pd.read_csv(f"data/{year}-ordinance.csv")
    dfYear["year"] = year
    df = pd.concat([df, dfYear], ignore_index=True)


# clean rows and columns
df["amount"] = df.apply(getRowAmount, axis=1)
df["fund_code"] = df.apply(lambda row: row["fund_code"] if len(row["fund_code"]) == 4 else "0" + row["fund_code"], axis=1)
df["appropriation_authority"] = df["appropriation_authority"].apply(lambda x: str(int(x)) if isinstance(x, float) and not math.isnan(x) else x)
df = df.drop(columns=["department", "_ordinance_amount_", "appropriation_ordinance"])

for colName in normalizedColumns:
    df[colName] = df[colName].str.upper()

df.tail()

,fund_type,fund_code,fund_description,department_number,department_description,appropriation_authority,appropriation_authority_description,appropriation_account,appropriation_account_description,amount,year
57023,GRANTS,925F,FEDERAL GRANT FUND,91,CHICAGO PUBLIC LIBRARY,280D,HUD - CPD - CPF - RUDY LOZANO BRANCH LIBRARY RENOVATION (14.251),909A,RESERVE BALANCE,2000000.0,2026
57024,GRANTS,925S,STATE GRANT FUND,91,CHICAGO PUBLIC LIBRARY,2842,SOS - CAPITAL CONSTRUCTION PROGRAM,909A,RESERVE BALANCE,10050000.0,2026
57025,GRANTS,925S,STATE GRANT FUND,91,CHICAGO PUBLIC LIBRARY,2895,SOS - LIBRARY DEVELOPMENT - PER CAPITA AND AREA,0005,SALARIES AND WAGES - ON PAYROLL,5020269.0,2026
57026,GRANTS,925S,STATE GRANT FUND,91,CHICAGO PUBLIC LIBRARY,2895,SOS - LIBRARY DEVELOPMENT - PER CAPITA AND AREA,0015,SCHEDULE SALARY ADJUSTMENTS,28808.0,2026
57027,GRANTS,925S,STATE GRANT FUND,91,CHICAGO PUBLIC LIBRARY,2895,SOS - LIBRARY DEVELOPMENT - PER CAPITA AND AREA,0044,FRINGE BENEFITS,3372923.0,2026


In [ ]:
# More Info from 2026 Budget Breakdown: https://public.tableau.com/app/profile/obm.data.analytics/viz/CityofChicago-BudgetataGlance/ChicagoBudgetataGlance?publish=yes
# Proceeds of debt issuances transferred between funds and reimbursements or internal transfers between funds need to be deducted to more accurately reflect the City appropriation.
# Total resources include revenues generated during the year.
budgets = pd.DataFrame({ "year": years, "budget": list(map(lambda x: int(getAnnualBudget(df, x)), years))})
budgets

,year,budget
0,2011,8700334637
1,2012,8614865000
2,2013,8745336000
3,2014,9114711000
4,2015,9555106000
5,2016,10056354000
6,2017,10677480000
7,2018,10752004000
8,2019,11400085000
9,2020,12607102000


In [ ]:
# More Info from Office of Inspector General: https://igchicago.org/information-portal/data-dashboards/city-budget-by-departments/
# Local funds are used by the City for non-capital operations with sources other than grant funds. These include the Corporate Fund, O’Hare Revenue Fund, Water Fund, and other similar funds. 
# Grants are financial awards given by the federal, state, or local government authority restricted for a specific project or service. 
# Community Development Block Grant (CDBG) funds are provided by federal and state governments to communities and people to be used in a variety of ways including housing, community development programs, healthy food initiatives, and sustainability services. 
funds = getValueCounts(["fund_type", "fund_code", "fund_description"])
funds.head(10)

,fund_type,fund_code,fund_description,count
0,LOCAL,0100,CORPORATE FUND,18603
1,LOCAL,0200,WATER FUND,4317
2,LOCAL,0300,VEHICLE FUND,2794
3,LOCAL,0610,MIDWAY AIRPORT FUND,2250
4,LOCAL,0314,SEWER FUND,2185
5,LOCAL,0740,O HARE REVENUE FUND,1852
6,LOCAL,0740,O'HARE REVENUE FUND,937
7,LOCAL,0740,CHICAGO O'HARE AIRPORT FUND,897
8,LOCAL,0346,LIBRARY FUND,871
9,LOCAL,0610,CHICAGO MIDWAY AIRPORT FUND,804


In [20]:
depts = getValueCounts(["department_number", "department_description"])
depts

,department_number,department_description,count
0,99,FINANCE GENERAL,4717
1,41,DEPARTMENT OF PUBLIC HEALTH,3040
2,50,FAMILY AND SUPPORT SERVICES,1878
3,31,DEPARTMENT OF LAW,1827
4,84,CDOT,1718
5,84,CHICAGO DEPT OF TRANSPORTATION,1641
6,50,DFSS,1541
7,27,FINANCE,1458
8,15,CITY COUNCIL,1390
9,57,DEPARTMENT OF POLICE,1365


In [9]:
authorities = getValueCounts(["appropriation_authority", "appropriation_authority_description"])
authorities.head()

,appropriation_authority,appropriation_authority_description,count
0,2005,FINANCE GENERAL,4481
1,2005,OFFICE OF INSPECTOR GENERAL,1425
2,2005,DEPARTMENT OF LAW,1188
3,2005,LAW,1073
4,2131,BUREAU OF ASSET MANAGEMENT,927


In [10]:
accounts = getValueCounts(["appropriation_account", "appropriation_account_description"])
accounts.head()

,appropriation_account,appropriation_account_description,count
0,140,PROFESSIONAL AND TECHNICAL SERVICES,2797
1,5,SALARIES AND WAGES - ON PAYROLL,2772
2,140,PROF & TECHNICAL SERVICES,1547
3,0140,FOR PROFESSIONAL AND TECHNICAL SERVICES AND OTHER THIRD PARTY BENEFIT AGREEMENTS,1378
4,44,FRINGE BENEFITS,1312


In [11]:
multFundCodes = getAltNames(funds, "fund_code", "fund_description")
multFundCodes.head()

,fund_code,count,names
0,0740,3,"[O HARE REVENUE FUND, O'HARE REVENUE FUND, CHICAGO O'HARE AIRPORT FUND]"
1,0521,3,"[LIB NOTE RED/INT-B, LIBRARY NOTE REDEMPTION AND INTEREST TENDER NOTES SERIES ""B"" FUND, LIBRARY PROPERTY TAX LEVY FUND]"
2,0510,3,"[BOND REDEMPTION AND INTEREST, BOND REDMPTN/IN F, BOND REDEMPTION AND INTEREST SERIES FUND]"
3,0681,3,"[MUNICIPAL ANNUNITY AND BENEFIT, MUNICIPAL EMPLOYEES' ANNUITY AND BENEFIT FUND, MUNICIPAL EMPLOYEE]"
4,0682,3,"[LABORERS' AND RETIREMENT BOARD ANNUITY AND BENEFIT FUND, LABORERS' ANNUNITY AND BENEFIT, LABORERS'/RETIREMNT]"


In [12]:
multDeptNums = getAltNames(depts, "department_number", "department_description")
multDeptNums.head()

,department_number,count,names
0,38,8,"[FLEET AND FACILITY MGMT, FLEET AND FACILITY MANAGEMENT, DEPARTMENT OF FLEET AND FACILITY MANAGEMENT, AIS, FFM, DEPARTMENT OF ASSETS INFORMATION AND SERVICES, DAIS, GENERAL SERVICES]"
1,73,5,"[COMM ANIMAL CARE AND CONTROL, CHICAGO ANIMAL CARE AND CONTROL, ANIMAL CARE / CONTROL, ANIMAL CARE AND CONTROL, CACC]"
2,41,4,"[DEPARTMENT OF PUBLIC HEALTH, CHICAGO DEPARTMENT OF PUBLIC HEALTH, HEALTH, CDPH]"
3,81,4,"[DEPT STREETS AND SANITATION, DSS, DEPARTMENT OF STREETS AND SANITATION, STREETS AND SANITATION]"
4,88,4,"[DEPT OF WATER MANAGEMENT, DWM, DEPARTMENT OF WATER MANAGEMENT, WATER MANAGEMENT]"


In [13]:
multAuthorities = getAltNames(authorities, "appropriation_authority", "appropriation_authority_description")
multAuthorities.head()

,appropriation_authority,count,names
0,2005,93,"[FINANCE GENERAL, OFFICE OF INSPECTOR GENERAL, DEPARTMENT OF LAW, LAW, CHICAGO FIRE DEPARTMENT, COMMISSIONER'S OFFICE, CHICAGO PUBLIC LIBRARY, FIRE DEPARTMENT, PLANNING AND DEVELOPMENT, FAMILY AND SUPPORT SERVICES, ELECTION AND ADMIN DIVISION, CITY CLERK, CITY TREASURER, OFFICE OF THE MAYOR, BUILDINGS, OFFICE OF PUBLIC SAFETY ADMINISTRATION, DEPARTMENT OF HUMAN RESOURCES, PROCUREMENT SERVICES, 2005 - FINANCE GENERAL, BACP, BOARD OF ETHICS, HUMAN RESOURCES, OFFICE OF CITY CLERK, DOIT, DEPT OF BUILDINGS, 2005 - DEPARTMENT OF LAW, ANIMAL CARE AND CONTROL, CITY COUNCIL, DEPARTMENT OF PLANNING AND DEVELOPMENT, DEPARTMENT OF PROCUREMENT SERVICES, OFFICE OF CITY TREASURER, ADMINISTRATIVE HEARINGS, PUBLIC SAFETY ADMINISTRATION, DEPARTMENT OF FAMILY AND SUPPORT SERVICES, BUS AFFAIRS AND CONSUMER PROT, DEPARTMENT OF BUSINESS AFFAIRS AND CONSUMER PROTECTION, LICENSE APPEAL COMMISSION, DEPARTMENT OF BUILDINGS, DEPARTMENT OF PROCUREMENT SERV, MOPD, OBM, COPA, ELECTION AND ADMINISTRATION DIVISION, POLICE BOARD, COMM ANIMAL CARE AND CONTROL, COMMUNITY COMMISSION FOR PUBLIC SAFETY AND ACCOUNTABILITY, IPRA, HUMAN RELATIONS, 2005 - OFFICE OF INSPECTOR GENERAL, CHICAGO ANIMAL CARE AND CONTROL, HOUSING AND ECONOMIC DEVELOPMT, CIVILIAN OFFICE OF POLICE ACCOUNTABILITY, 2005 - FIRE DEPARTMENT, DEPT OF ADMINISTRATIVE HEARING, CHICAGO POLICE BOARD, OFFICE FOR PEOPLE WITH DISABIL, 2005 - CITY COMPTROLLER, OFFICE OF BUDGET & MANAGEMENT, DEPARTMENT OF ADMINISTRATIVE HEARINGS, CITY TREASURER'S OFFICE, COMMISSION ON HUMAN RELATIONS, OFFICE OF BUDGET AND MANAGEMENT, 2005 - DEPT OF BUILDINGS, MAYOR'S OFFICE FOR PEOPLE WITH DISABILITIES, CHICAGO COMMISSION ON HUMAN RELATIONS, 2005 - COMMISSIONER'S OFFICE, 2005 - DEPARTMENT OF HUMAN RESOURCES, 2005 - HOUSING AND ECONOMIC DEVELOPMT, 2005 - FAMILY AND SUPPORT SERVICES, 2005 - BUS AFFAIRS AND CONSUMER PROT, 2005 - DEPARTMENT OF ENVIRONMENT, 2005 - DEPARTMENT OF PROCUREMENT SERV, 2005 - CHICAGO PUBLIC LIBRARY, 2005 - ELECTION AND ADMIN DIVISION, 2005 - CITY CLERK, 2005 - COMM ANIMAL CARE AND CONTROL, 2005 - DOIT, 2005 - OFFICE OF COMPLIANCE, 2005 - BUREAU OF CULTURAL AFFAIRS, 2005 - CITY TREASURER, 2005 - DEPT OF ADMINISTRATIVE HEARING, 2005 - IPRA, 2005 - COMMISSION ON HUMAN RELATIONS, 2005 - OFFICE OF BUDGET & MANAGEMENT, 2005 - OFFICE FOR PEOPLE WITH DISABIL, 2005 - OFFICE OF THE MAYOR, 2005 - BOARD OF ETHICS, 2005 - CITY COUNCIL, 2005 - POLICE BOARD, 2005 - LICENSE APPEAL COMMISSION, CCPSA, 2005 - COMMISSIONERS OFFICE, COMMISSIONERS OFFICE]"
1,2800,30,"[INNOVATION DELIVERY GRANT, AMPLIFIED PHONES PROGRAM, CENTRAL GRANTS MANAGEMENT, DHS ACCOUNTING, O'HARE - FAA (MOA) - PHASE II, CTY - GRANTS MANAGEMENT, 2800 - AVIATION ENVIRONMENTAL AND RE, MICD, NEIGHBORHOOD STABILIZATION PRO, HAZARD MITIGATION PROGRAM, 2800 - DHS ACCOUNTING, HEALTH IT COORDINATOR, 2800 - TRAFFIC SIGNALS/STREET LIGHTS, 2800 - EDGEWATER BRANCH - CAPITAL, 2800 - MICD, 2800 - O'HARE - FAA (MOA) - PHASE II, 2800 - ARRA - CHICAGO ALT. FUELS, 2800 - HEALTH IT COORDINATOR, 2800 - GREAT LAKES RESTORATION, 2800 - NEIGHBORHOOD STABILIZATION PRO, 2800 - ARRA PORT SECURITY GRANT PROGR, 2800 - LABOR MGMT. HEALTH CARE SAVING, EDGEWATER BRANCH - CAPITAL, URBAN BIRD TREATY, ARRA PORT SECURITY GRANT PROGR, HELP AMERICA VOTE ELECTION SEC, HEALTH DISPARITIES CHICAGO, TREE PLANTING, STATE URBAN FORESTRY RESILIENCE (TREE PLANTING), CLIR RECORDINGS AT RISK PROGRAM]"
2,2505,26,"[ADMINISTRATION AND MONITORING, FINANCE AND ADMINISTRATION, DISABILITY RESOURCES, EDUCATION OUTREACH AND INTERG, TROUBLED BUILDINGS PROGRAM, ENVIRONMENTAL REVIEW, PLANNING AND ADMINISTRATION, 2505 - FINANCE AND ADMINISTRATION, EDUCATION OUTREACH AND INTERGROUP, COMMUNITY ENHANCEMENT, PLANNING AND ADMINISTRATION, LEGAL SERVICES, 2505 - ADMINISTRATION AND MONITORING, 2505 - EDUCATION OUTREACH AND INTERG, 2505 - DISABILITY RESOURCES, 2505 - SPECIAL ACCOUNTING DIVISION, HUD - CPD - CDBG - PLANNING AND ADMINISTRATION (14.218), 2505 - TROUBLED BUILDIN

In [14]:
multAccounts = getAltNames(accounts,"appropriation_account", "appropriation_account_description")
multAccounts.head()

,appropriation_account,count,names
0,9438,4,"[REIMBURSEMENT - 2FM, REIMBURSEMENT - DGS, REIMBURSEMENT - AIS, REIMBURSEMENT - DAIS]"
1,9771,4,"[TRANSFER FOR SERVICES - 2FM, TRANSFER FOR SERVICES - AIS, TRANSFER FOR SERVICES 2FM, TRANSFER FOR SERVICES - DAIS]"
2,190,3,"[TELEPHONE - NON-CENTREX BILLINGS, TELEPHONE-CENTREX, TELEPHONE - CENTREX BILLINGS]"
3,989,3,"[REFUND - CANCELLED VOUCHER, REFUND CANCELLED VOUCHR, REFUND CANCELLED VOUCHER]"
4,17,3,"[CITY COUNCIL - SALARIED EMPLOYEES, CITY COUNCIL-SALARY INC, WARD STAFF WAGE ALLOWANCE]"
